## JSONL Generation

In [ ]:
### Generating JSONL using eviction decisions

In [ ]:
### Generating JSONL using few-shot examples
import pandas as pd
import json

df = pd.read_csv("../data/cache_examples - list.csv")
df = df.drop(columns=["Type"])
df.columns = ["prompt", "response"]

# Generate messages-based format
jsonl_data = []

for _, row in df.iterrows():
    prompt = str(row['prompt']).strip()
    response = str(row['response']).strip()

    jsonl_data.append({
        "messages": [
            {"role": "system", "content": "You are a helpful assistant specialized in cache replacement analysis."},
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response}
        ]
    })

# Save to JSONL file
output_file = "../data/openai_finetune_data_v2.jsonl"
with open(output_file, "w") as f:
    for entry in jsonl_data:
        f.write(json.dumps(entry) + "\n")

print(f"✅ Saved {len(jsonl_data)} examples to '{output_file}'")

## Open-AI API Finetuning

In [ ]:
import json

input_file = "openai_training_data_with_program_semantics.jsonl"
output_file = "limited_1000_examples.jsonl"
max_samples = 1000

limited_data = []

# Read and load up to 100 lines
with open(input_file, "r") as f:
    for i, line in enumerate(f):
        if i >= max_samples:
            break
        item = json.loads(line)
        limited_data.append(item)

# Save the limited data back to a new JSONL file
with open(output_file, "w") as f:
    for item in limited_data:
        f.write(json.dumps(item) + "\n")

print(f"Saved {len(limited_data)} examples to {output_file}")


In [ ]:
import openai

openai.api_key = ""

# Upload file
file_response = openai.files.create(
    file=open("../data/openai_finetune_data_v2.jsonl", "rb"),
    purpose="fine-tune"
)

file_id = file_response.id
print("Uploaded file ID:", file_id)

In [ ]:
# Create fine-tuning job

fine_tune_response = openai.fine_tuning.jobs.create(
    training_file=file_id,
    model="gpt-4o-mini-2024-07-18",  # Make sure this version matches availability
    suffix="cache-replacement-finetuning-few-shot-examples",
)

print("Job ID:", fine_tune_response.id)
print("Fine-tuning job created successfully.")

In [ ]:
# MONITOR finetuining job status
# List recent jobs
jobs = openai.fine_tuning.jobs.list(limit=3)
for job in jobs.data:
    print(job.id, job.status)

# Or track a specific job
job_id = fine_tune_response.id
job_status = openai.fine_tuning.jobs.retrieve(job_id)
print(job_status.status)